The goal of this homework is to create a regression model for predicting the car fuel efficiency (column 'fuel_efficiency_mpg').

In [4]:
!wget -nc https://raw.githubusercontent.com/k-candidate/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

File ‘car_fuel_efficiency_2026.csv’ already there; not retrieving.



In [32]:
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [6]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')

## Preparing the dataset

Preparation:
- Fill missing values with zeros.
- Do the split
- Use DictVectorizer(sparse=True) to turn the dataframes into matrices.

In [10]:
df.shape

(10000, 11)

In [11]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [12]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [13]:
df = df.fillna(0)

In [14]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

## Question 1

Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable.
- Train a model with `max_depth=1`.

Which original feature is used for splitting the data? If you use `DictVectorizer`, map a one-hot feature such as `origin=USA` back to its original column name.
- `'vehicle_weight'`
- `'model_year'`
- `'origin'`
- `'fuel_type'`

For the training we can use https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html  
And then to see the tree we can use https://scikit-learn.org/stable/modules/generated/sklearn.tree.export_text.html

In [17]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with 

In [18]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



Answer: `'model_year'`

## Question 2

Train a random forest regressor with these parameters:
- `n_estimators=10`
- `random_state=1`
- `n_jobs=-1` (optional - to make training faster)

What's the RMSE of this model on the validation data?
- 0.1837
- 1.837
- 18.37
- 183.7

For the training we can use https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html  

And for the RMSE we can get it from a previous chapter: https://github.com/k-candidate/Machine-Learning-Zoomcamp-2026/blob/main/02_-_Regression/02-notes.ipynb

```python
def rmse(y, y_pred):
    """
    y: vector of actual values
    y_pred: vector of predicted values
    """
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)
```

We can calculate the `mse` as we did in the function, or we could rely on https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html which will give us this:

```python
def rmse(y, y_pred):
    """
    y: vector of actual values
    y_pred: vector of predicted values
    """
    mean_squared_error(y, y_pred)
    return np.sqrt(mse)
```

Or we could write even less by just using directly https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html:

```python
root_mean_squared_error(y, y_pred)
```

In [36]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
# n_estimators: the number of trees in the forest.
# n_jobs: the number of jobs to run in parallel. -1 means using all processors.
# random_state: acts as a seed for the internal pseudo-random number generator used during model training. 
# Setting random_state (to any integer) controls this randomness to make the code reproducible.
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [37]:
score = root_mean_squared_error(y_val, rf.predict(X_val))
score, round(score, 3)

(1.8332197358745623, 1.833)

We got an RMSE of 1.833. The closest number from the possible answers is 1.837.  
These numbers can change due to different versions of libraries.

Answer: 1.837

## Question 3

Now let's experiment with the `n_estimators` parameter

- Try these values of this parameter: `[10, 50, 100, 150]`.
- Set `random_state` to `1`.
- Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.
- 10
- 50
- 100
- 150

In [38]:
scores = {}
for n in [10, 50, 100, 150]:
    m = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    m.fit(X_train, y_train)
    s = root_mean_squared_error(y_val, m.predict(X_val))
    scores[n] = s
    print(f'n_estimators={n:<5} rmse={s:.6f} -> {round(s, 3)}')

n_estimators=10    rmse=1.833220 -> 1.833
n_estimators=50    rmse=1.773988 -> 1.774
n_estimators=100   rmse=1.769786 -> 1.77
n_estimators=150   rmse=1.769198 -> 1.769


Answer: 150

## Question 4

Let's select the best `max_depth`:

- Try different values of `max_depth`: `[10, 15, 20, 25]`
- For each of these values,
  - try different values of `n_estimators` from `[10, 50, 100, 150]`
  - calculate the mean RMSE
- Fix the random seed: `random_state=1`

What's the best `max_depth`, using the mean RMSE?
- 10
- 15
- 20
- 25

In [39]:
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        m = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        # max_depth: the maximum depth of the tree.
        m.fit(X_train, y_train)
        scores.append(root_mean_squared_error(y_val, m.predict(X_val)))
    print(f'max_depth={d:<4} mean_rmse={np.mean(scores):.6f} -> {round(np.mean(scores), 3)}'
          f'   per-n={[round(s, 3) for s in scores]}')

max_depth=10   mean_rmse=1.754392 -> 1.754   per-n=[1.782, 1.748, 1.744, 1.744]
max_depth=15   mean_rmse=1.784224 -> 1.784   per-n=[1.831, 1.771, 1.768, 1.767]
max_depth=20   mean_rmse=1.786771 -> 1.787   per-n=[1.837, 1.772, 1.769, 1.768]
max_depth=25   mean_rmse=1.786305 -> 1.786   per-n=[1.837, 1.772, 1.768, 1.768]


Answer: 10

## Question 5

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the `feature_importances_` field.

For this homework question, we'll find the most important feature:

- Train the model with these parameters:
  - `n_estimators=10`,
  - `max_depth=20`,
  - `random_state=1`,
  - `n_jobs=-1` (optional)
- Get the feature importance information from this model

What's the most important feature (among these 4)?
- `vehicle_weight`
- `horsepower`
- `acceleration`
- `engine_displacement`

In [40]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_l

In [45]:
# feature_importances_ is an attribute of the model we just trained.
# We can see it in the command below.
# Or we can see it in the UI if you are in the jupyter web ui or in vscode with the jupyter extension. It does not show in GH web ui.
# The attribute is mentioned in the official doc: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
rf.feature_importances_

array([0.05499763, 0.00795063, 0.02006223, 0.00563848, 0.06128144,
       0.01657053, 0.16071245, 0.01627193, 0.07827125, 0.29610715,
       0.00731824, 0.01940147, 0.00651495, 0.00343644, 0.04215174,
       0.20331344])

In [43]:
# We can see above that we need to tie each "feature importance" to its "feature". For that we need the dictVector. 
# We can display it in any way we want: a dictionary, or a list of tuples, or a dataframe ...

importance = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False)

importance

,feature,importance
9,model_year,0.296107
15,vehicle_weight,0.203313
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078271
4,engine_displacement,0.061281
0,acceleration,0.054998
14,origin=USA,0.042152
2,drivetrain=Front-wheel drive,0.020062
11,num_doors,0.019401
5,fuel_type=Diesel,0.016571


In [44]:
importance[importance.feature.isin(
    ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement'])]

,feature,importance
15,vehicle_weight,0.203313
8,horsepower,0.078271
4,engine_displacement,0.061281
0,acceleration,0.054998


Answer: `vehicle_weight`

## Question 6

Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:

- Install XGBoost
- Create DMatrix for train and validation
- Create a watchlist
- Train a model with these parameters for 100 rounds:

```python
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
```

Now change `eta` from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?
- 0.3
- 0.1
- Both give equal value

In [46]:
features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]

In [47]:
for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=25)

    print(f'eta={eta}:  train_rmse={root_mean_squared_error(y_train, model.predict(dtrain)):.6f}'
          f'   val_rmse={root_mean_squared_error(y_val, model.predict(dval)):.6f}')
    print()

[0]	train-rmse:2.40653	val-rmse:2.46782
[25]	train-rmse:1.35837	val-rmse:1.74359
[50]	train-rmse:1.18645	val-rmse:1.77517
[75]	train-rmse:1.02727	val-rmse:1.80682
[99]	train-rmse:0.90106	val-rmse:1.82646
eta=0.3:  train_rmse=0.901056   val_rmse=1.826458

[0]	train-rmse:2.74739	val-rmse:2.78635
[25]	train-rmse:1.50918	val-rmse:1.72751
[50]	train-rmse:1.40368	val-rmse:1.71358
[75]	train-rmse:1.35099	val-rmse:1.72057
[99]	train-rmse:1.31192	val-rmse:1.72481
eta=0.1:  train_rmse=1.311918   val_rmse=1.724810



Answer: 0.1

<style>
/* 1. Forces wide tables and standard HTML outputs to fit */
.output_subarea, .output_html, table.dataframe {
    max-width: 100% !important;
    overflow-x: auto !important;
    display: block !important;
}

/* 2. Forces standard stdout code prints, stderr outputs (like wget), 
      and Python Traceback errors to wrap text safely */
div.output_area pre, 
div.output_subarea.output_stream.output_stderr pre, 
div.output_subarea.output_error pre {
    white-space: pre-wrap !important;
    word-break: break-word !important;
}
</style>
